In [10]:
from datasets import load_dataset
from transformers import BertTokenizer
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/amananandrai/ag-news-classification-dataset/train.csv")
tokenizer = BertTokenizer.from_pretrained("/kaggle/input/models/ahsan56/bert/transformers/default/1/bert-base-uncased")
dataset = Dataset.from_pandas(df)
def tokenize_function(examples):
    return tokenizer(examples["Title"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

In [2]:
from datasets import Dataset
from transformers import BertTokenizer
import pandas as pd


df = pd.read_csv("/kaggle/input/datasets/amananandrai/ag-news-classification-dataset/train.csv")


df['text'] = "Title: " + df['Title'] + " Content: " + df['Description']


df['label'] = df['Class Index'] - 1


dataset = Dataset.from_pandas(df[['text', 'label']])


tokenizer = BertTokenizer.from_pretrained("/kaggle/input/models/ahsan56/bert/transformers/default/1/bert-base-uncased")


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

In [3]:
from transformers import BertForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score


model = BertForSequenceClassification.from_pretrained(
    "/kaggle/input/models/ahsan56/bert/transformers/default/1/bert-base-uncased", 
    num_labels=4
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

dataset_split = tokenized_datasets.train_test_split(test_size=0.2)
train_dataset = dataset_split['train']
val_dataset = dataset_split['test']

from transformers import TrainingArguments, Trainer


training_args = TrainingArguments(
    output_dir="./bert_news_results",
    eval_strategy="epoch",            
    save_strategy="epoch",
    learning_rate=2e-5,               
    per_device_train_batch_size=16,   
    per_device_eval_batch_size=16,
    num_train_epochs=3,               
    weight_decay=0.01,
    load_best_model_at_end=True,      
    report_to="none"                  
)


from transformers import DataCollatorWithPadding


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator, # Add this line
)


trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /kaggle/input/models/ahsan56/bert/transformers/default/1/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized b

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.513985,0.499342,0.916125,0.915955
2,0.351845,0.485732,0.921792,0.921618
3,0.241361,0.549184,0.923250,0.923207


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=9000, training_loss=0.4084503648546007, metrics={'train_runtime': 4307.3927, 'train_samples_per_second': 66.862, 'train_steps_per_second': 2.089, 'total_flos': 1.6749901226702592e+16, 'train_loss': 0.4084503648546007, 'epoch': 3.0})

In [4]:
model.save_pretrained("./fine_tuned_bert_agnews")
tokenizer.save_pretrained("./fine_tuned_bert_agnews")
print("Training Complete! Model saved to ./fine_tuned_bert_agnews")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Complete! Model saved to ./fine_tuned_bert_agnews


In [5]:
import shutil
shutil.make_archive("my_bert_model", 'zip', "./fine_tuned_bert_agnews")


'/kaggle/working/my_bert_model.zip'